In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import os
os.chdir('C:\Kaggle_Competition\Playground\S6E4-Irrigation-Need')
import json
import warnings
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.inspection import permutation_importance
from src.utils import (
    read_csv_file, 
    save_csv_file, 
    agnostic_bacc_scorer,
    compute_bacc,
    compute_logloss,
    compute_recall_per_class
)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import mlflow
from tqdm.notebook import tqdm
from pytabkit import (
	RealMLP_TD_Classifier,
	TabM_D_Classifier,
	Resnet_RTDL_D_Classifier,
	CatBoost_TD_Classifier,
	LGBM_TD_Classifier,
	XGB_TD_Classifier
)
from src.fe import *

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

### Prepare data

In [10]:
train_raw = read_csv_file(r'data\raw\train.csv')
test_raw = read_csv_file(r'data\raw\test.csv')

# Train and test ids
train_ids = train_raw['id']
test_ids = test_raw['id']

n_classes = 3

Reading file from: data\raw\train.csv
Reading file from: data\raw\test.csv


In [11]:
# X and y split
X = train_raw.drop(['id', 'irrigation_need'], axis=1)
y = train_raw['irrigation_need'].map(config.TARGET_MAP)

# Test data
test_data = test_raw.drop('id', axis=1)

In [12]:
# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [13]:
RUN = 'v1'

### 1. Catboost Core

In [ ]:
EXP_NAME = "CATBOOST"
RUN_NAME = f"CATBOOST_seed{config.SEED}_{RUN}"

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    # ---------------- PARAMS ----------------
    mlflow.log_params({
        "seed": config.SEED,
        "n_folds": config.N_FOLDS,
        "n_rows": len(X),
        "n_cols": X.shape[1],
        "n_cat_cols": len(cat_cols),
        "n_num_cols": len(num_cols),
        **config.CATBOOST_PARAMS
    })

    mlflow.log_text("\n".join(X.columns.tolist()), "artifacts/all_feature_names.txt")
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")

    dtype_map = {col: str(dtype) for col, dtype in X.dtypes.items()}
    mlflow.log_dict(dtype_map, "artifacts/feature_dtypes.json")

    # ---------------- CV SETUP ----------------
    skf = StratifiedKFold(
        n_splits=config.N_FOLDS,
        shuffle=True,
        random_state=config.SEED
    )

    test_pool = cb.Pool(test_data, cat_features=cat_cols)
    n_classes = len(np.unique(y))

    final_oof_preds = np.zeros((len(X), n_classes))
    final_test_preds = np.zeros((len(test_data), n_classes))
    feature_importance = []

    bacc_scores = []
    ll_scores = []

    # ---------------- TRAINING ----------------
    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(skf.split(X, y), start=1),
        colour='green',
        ncols=300,
        total=config.N_FOLDS,
        desc='Model Training'
    ):

        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):

            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

            train_pool = cb.Pool(X_train, y_train, cat_features=cat_cols)
            valid_pool = cb.Pool(X_valid, y_valid, cat_features=cat_cols)

            model = cb.train(
                train_pool,
                config.CATBOOST_PARAMS,
                eval_set=valid_pool,
                verbose=False
            )

            # Log model
            mlflow.catboost.log_model(model, name='model')

            # predictions
            oof_preds = model.predict(valid_pool, prediction_type="Probability")
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict(test_pool, prediction_type="Probability")
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=3,
                n_jobs=-1,
                random_state=config.SEED
            )

            feature_importance.append(fi.importances_mean)
            
			# ---------------- CLEANUP ----------------
            del model, train_pool, valid_pool, oof_preds, test_preds, fi
            gc.collect()

    # ---------------- FINAL METRICS ----------------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    bacc_std = np.std(bacc_scores)
    ll_std = np.std(ll_scores)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    mlflow.log_metrics({
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": bacc_std,
        "std_logloss": ll_std,
        "recall_class_0": recall_per_class[0],
        "recall_class_1": recall_per_class[1],
        "recall_class_2": recall_per_class[2],
    })

    # ---------------- FEATURE IMPORTANCE ----------------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)
    fi_df = pd.DataFrame({
        "feature": X.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")

    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)

# ---------- Saving Files ----------

# OOF probabilities
oof_df = pd.DataFrame(
    final_oof_preds,
    columns=["0", "1", "2"]
)
oof_df.insert(0, "id", train_ids)

save_csv_file(
    oof_df,
    os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv")
)

# Test probabilities
test_df = pd.DataFrame(
    final_test_preds,
    columns=["0", "1", "2"]
)
test_df.insert(0, "id", test_ids)

save_csv_file(
    test_df,
    os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
)

### 2. Lightgbm Core

In [ ]:
EXP_NAME = "LGBM"
RUN_NAME = f"LGBM_seed{config.SEED}_{RUN}"

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.log_params({
        "seed": config.SEED,
        "n_folds": config.N_FOLDS,
        "n_rows": len(X),
        "n_cols": X.shape[1],
        "n_cat_cols": len(cat_cols),
        "n_num_cols": len(num_cols),
        **config.LGBM_PARAMS
    })

    mlflow.log_text("\n".join(X.columns.tolist()), "artifacts/all_feature_names.txt")
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")

    dtype_map = {col: str(dtype) for col, dtype in X.dtypes.items()}
    mlflow.log_dict(dtype_map, "artifacts/feature_dtypes.json")

    skf = StratifiedKFold(
        n_splits=config.N_FOLDS,
        shuffle=True,
        random_state=config.SEED
    )

    n_classes = len(np.unique(y))

    final_oof_preds = np.zeros((len(X), n_classes))
    final_test_preds = np.zeros((len(test_data), n_classes))
    feature_importance = []

    bacc_scores = []
    ll_scores = []

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(skf.split(X, y), start=1),
        colour='green',
        ncols=300,
        total=config.N_FOLDS,
        desc='Model Training'
    ):

        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):

            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
            
            X_train[cat_cols] = X_train[cat_cols].astype('category')
            X_valid[cat_cols] = X_valid[cat_cols].astype('category')

            train_set = lgb.Dataset(
                X_train,
                label=y_train,
                categorical_feature=cat_cols,
            )
            valid_set = lgb.Dataset(
                X_valid,
                label=y_valid,
                categorical_feature=cat_cols,
                reference=train_set,
            )

            model = lgb.train(
                params=config.LGBM_PARAMS,
                train_set=train_set,
                valid_sets=[valid_set]
            )

            mlflow.lightgbm.log_model(model, name="model")

            oof_preds = model.predict(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_data[cat_cols] = test_data[cat_cols].astype('category')
            test_preds = model.predict(test_data)
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=3,
                n_jobs=-1,
                random_state=config.SEED
            )

            feature_importance.append(fi.importances_mean)
            
			# ---------------- CLEANUP ----------------
            del model, train_set, valid_set, oof_preds, test_preds, fi
            gc.collect()

    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    bacc_std = np.std(bacc_scores)
    ll_std = np.std(ll_scores)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    mlflow.log_metrics({
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": bacc_std,
        "std_logloss": ll_std,
        "recall_class_0": recall_per_class[0],
        "recall_class_1": recall_per_class[1],
        "recall_class_2": recall_per_class[2],
    })

    fi_mean = np.mean(np.vstack(feature_importance), axis=0)
    fi_df = pd.DataFrame({
        "feature": X.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")

    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)

# ---------- Saving Files ----------

# OOF probabilities
oof_df = pd.DataFrame(
    final_oof_preds,
    columns=["0", "1", "2"]
)
oof_df.insert(0, "id", train_ids)

save_csv_file(
    oof_df,
    os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv")
)

# Test probabilities
test_df = pd.DataFrame(
    final_test_preds,
    columns=["0", "1", "2"]
)
test_df.insert(0, "id", test_ids)

save_csv_file(
    test_df,
    os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
)

### 3. HistGBM

In [ ]:
EXP_NAME = "HGB"
RUN_NAME = f"HGB_seed{config.SEED}_{RUN}"

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.log_params({
        "seed": config.SEED,
        "n_folds": config.N_FOLDS,
        "n_rows": len(X),
        "n_cols": X.shape[1],
        "n_cat_cols": len(cat_cols),
        "n_num_cols": len(num_cols),
        **config.HISTGBM_PARAMS
    })

    mlflow.log_text("\n".join(X.columns.tolist()), "artifacts/all_feature_names.txt")
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")

    dtype_map = {col: str(dtype) for col, dtype in X.dtypes.items()}
    mlflow.log_dict(dtype_map, "artifacts/feature_dtypes.json")

    skf = StratifiedKFold(
        n_splits=config.N_FOLDS,
        shuffle=True,
        random_state=config.SEED
    )

    n_classes = len(np.unique(y))

    final_oof_preds = np.zeros((len(X), n_classes))
    final_test_preds = np.zeros((len(test_data), n_classes))
    feature_importance = []

    bacc_scores = []
    ll_scores = []

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(skf.split(X, y), start=1),
        colour='green',
        ncols=300,
        total=config.N_FOLDS,
        desc='Model Training'
    ):

        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):

            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
            
            X_train[cat_cols] = X_train[cat_cols].astype('category')
            X_valid[cat_cols] = X_valid[cat_cols].astype('category')

            model = HistGradientBoostingClassifier(**config.HISTGBM_PARAMS)

            model.fit(X_train, y_train)

            mlflow.sklearn.log_model(model, name="model")

            # predictions
            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")
            
            test_data[cat_cols] = test_data[cat_cols].astype('category')
            test_preds = model.predict_proba(test_data)
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=3,
                n_jobs=-1,
                random_state=config.SEED
            )

            feature_importance.append(fi.importances_mean)
            
			# ---------------- CLEANUP ----------------
            del model, oof_preds, test_preds, fi
            gc.collect()

    # ---------------- FINAL METRICS ----------------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    bacc_std = np.std(bacc_scores)
    ll_std = np.std(ll_scores)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    mlflow.log_metrics({
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": bacc_std,
        "std_logloss": ll_std,
        "recall_class_0": recall_per_class[0],
        "recall_class_1": recall_per_class[1],
        "recall_class_2": recall_per_class[2],
    })

    # ---------------- FEATURE IMPORTANCE ----------------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)
    fi_df = pd.DataFrame({
        "feature": X.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")

    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)


# ---------- Saving Files ----------

# OOF probabilities
oof_df = pd.DataFrame(
    final_oof_preds,
    columns=["0", "1", "2"]
)
oof_df.insert(0, "id", train_ids)

save_csv_file(
    oof_df,
    os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv")
)

# Test probabilities
test_df = pd.DataFrame(
    final_test_preds,
    columns=["0", "1", "2"]
)
test_df.insert(0, "id", test_ids)

save_csv_file(
    test_df,
    os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
)